# Data Cleaning & Canonicalization

Transform the raw JSON records into the finalized canonical product schema.

In [1]:
!git clone https://github.com/jnDhruv/multi-modal-fashion-ecom
!ls

Cloning into 'multi-modal-fashion-ecom'...
remote: Enumerating objects: 367, done.
remote: Counting objects: 100% (367/367), done.
remote: Compressing objects: 100% (323/323), done.
remote: Total 367 (delta 40), reused 322 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (367/367), 5.26 MiB | 16.83 MiB/s, done.
Resolving deltas: 100% (40/40), done.
multi-modal-fashion-ecom  __notebook__.ipynb


In [2]:
import sys
from pathlib import Path
import importlib

import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/kaggle/working/multi-modal-fashion-ecom")
SCRIPTS_DIR = PROJECT_ROOT / "data" / "scripts"

sys.path.append(str(SCRIPTS_DIR))

In [3]:
!pip install -q -r /kaggle/working/multi-modal-fashion-ecom/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 24.3 MB/s eta 0:00:00


In [4]:
import pandas
import ftfy
import bs4
import pyarrow

print("Dependencies OK")

Dependencies OK


In [5]:
import clean_data

print(clean_data.__file__)

/kaggle/working/multi-modal-fashion-ecom/data/scripts/clean_data.py


In [6]:
DATASET_DIR = Path(
    "/kaggle/input/datasets/paramaggarwal/"
    "fashion-product-images-dataset/fashion-dataset"
)

STYLES_CSV = DATASET_DIR / "styles.csv"

valid_ids = clean_data.load_styles_csv_ids(STYLES_CSV)

print("Valid IDs:", len(valid_ids))
print("Sample IDs:", list(valid_ids)[:10])

Valid IDs: 44424
Sample IDs: [1163, 1164, 1165, 1525, 1526, 1528, 1529, 1530, 1531, 1532]


In [7]:
STYLES_DIR = DATASET_DIR / "styles"

json_records = clean_data.load_json_records(STYLES_DIR)

print("Loaded JSON records:", len(json_records))

Loaded JSON records: 44446


In [8]:
clean_records = clean_data.filter_valid_records(
    json_records,
    valid_ids
)

print("Original JSON records:", len(json_records))
print("Clean records:", len(clean_records))
print("Dropped:", len(json_records) - len(clean_records))

Original JSON records: 44446
Clean records: 44424
Dropped: 22


In [9]:
clean_ids = {
    record["data"]["id"]
    for record in clean_records
}

print("Unique clean IDs:", len(clean_ids))
print("All IDs valid:", clean_ids.issubset(valid_ids))
print("Missing valid IDs:", len(valid_ids - clean_ids))

Unique clean IDs: 44424
All IDs valid: True
Missing valid IDs: 0


In [10]:
# Test extract_type_name
print("1. extract_type_name")
print(clean_data.extract_type_name({
    "id": 90,
    "typeName": "Tshirts"
}))
print(clean_data.extract_type_name("Men"))
print(clean_data.extract_type_name(None))


# Test missing attribute detection
print("\n2. is_missing_attribute")
for value in [None, "", "NA", "N/A", "Regular Fit", "Cotton"]:
    print(repr(value), "->", clean_data.is_missing_attribute(value))


# Test descriptor unwrapping
print("\n3. unwrap_descriptor")
sample_descriptor = {
    "description": {
        "descriptorType": "description",
        "value": "<p>Test <strong>shirt</strong></p>"
    }
}
print(
    clean_data.unwrap_descriptor(
        sample_descriptor,
        "description"
    )
)


# Test encoding
print("\n4. fix_encoding")
print(clean_data.fix_encoding("30Â° C machine wash"))
print(clean_data.fix_encoding('32â€�'))


# Test HTML stripping
print("\n5. strip_html")
print(
    clean_data.strip_html(
        "<p>Regular <strong>fit</strong><br>cotton shirt</p>"
    )
)


# Test truncation
print("\n6. truncate_text")
long_text = "This is a test sentence " * 100
truncated = clean_data.truncate_text(long_text)

print("Original length:", len(long_text))
print("Truncated length:", len(truncated))
print("Ends cleanly:", not truncated.endswith(" "))


# Test complete description cleaning
print("\n7. clean_description")
raw_description = """
<div>
    <strong>Style Note</strong><br>
    This is a <b>30Â° C</b> cotton shirt.
</div>
"""

print(clean_data.clean_description(raw_description))

1. extract_type_name
Tshirts
Men
None

2. is_missing_attribute
None -> True
'' -> True
'NA' -> True
'N/A' -> True
'Regular Fit' -> False
'Cotton' -> False

3. unwrap_descriptor
<p>Test <strong>shirt</strong></p>

4. fix_encoding
30° C machine wash
32�

5. strip_html
Regular fit cotton shirt

6. truncate_text
Original length: 2400
Truncated length: 1497
Ends cleanly: True

7. clean_description
Style Note This is a 30° C cotton shirt.


In [11]:
sample_row = clean_data.build_row(clean_records[1000])

sample_row

{'id': 26233,
 'product_display_name': 'Proline Men Red Polo T-shirt',
 'brand_name': 'Proline',
 'gender': 'Men',
 'master_category': 'Apparel',
 'sub_category': 'Topwear',
 'article_type': 'Tshirts',
 'base_colour': 'Red',
 'season': 'Summer',
 'usage': 'Casual',
 'year': 2012,
 'price': 599,
 'discounted_price': 479,
 'description': 'Red polo T-shirt, has a polo collar, short button placket, short sleeves',
 'image_url': 'http://assets.myntassets.com/v1/images/style/properties/778605c0c9603e03c84bf7f36d9756e4_images.jpg',
 'pattern': 'Solid',
 'fabric': 'Polyester',
 'sleeve_length': 'Short Sleeves',
 'occasion': 'Casual',
 'fit': 'Regular Fit',
 'neck': 'Polo Collar',
 'length': 'Regular'}

In [12]:
for key, value in sample_row.items():
    print(f"{key}: {value}")

id: 26233
product_display_name: Proline Men Red Polo T-shirt
brand_name: Proline
gender: Men
master_category: Apparel
sub_category: Topwear
article_type: Tshirts
base_colour: Red
season: Summer
usage: Casual
year: 2012
price: 599
discounted_price: 479
description: Red polo T-shirt, has a polo collar, short button placket, short sleeves
image_url: http://assets.myntassets.com/v1/images/style/properties/778605c0c9603e03c84bf7f36d9756e4_images.jpg
pattern: Solid
fabric: Polyester
sleeve_length: Short Sleeves
occasion: Casual
fit: Regular Fit
neck: Polo Collar
length: Regular


In [13]:
sample_search_text = clean_data.build_search_text(sample_row)

print(sample_search_text)

Proline Men Red Polo T-shirt Proline Apparel Topwear Tshirts Red Casual Solid Polyester Short Sleeves Casual Regular Fit Polo Collar Regular Red polo T-shirt, has a polo collar, short button placket, short sleeves


In [14]:
rows = [
    clean_data.build_row(record)
    for record in clean_records
]

products_df = pd.DataFrame(rows)

products_df["search_text"] = products_df.apply(
    lambda row: clean_data.build_search_text(row.to_dict()),
    axis=1
)

In [15]:
print("Rows:", len(products_df))
print("Columns:", len(products_df.columns))
print(products_df.columns.tolist())

Rows: 44424
Columns: 23
['id', 'product_display_name', 'brand_name', 'gender', 'master_category', 'sub_category', 'article_type', 'base_colour', 'season', 'usage', 'year', 'price', 'discounted_price', 'description', 'image_url', 'pattern', 'fabric', 'sleeve_length', 'occasion', 'fit', 'neck', 'length', 'search_text']


*Note: styleImages has multiple image-key variants; default is preferred, with front/left/top fallbacks.*
Removing the products entirely from the dataset

In [16]:
missing_image_rows = products_df[
    products_df["image_url"].isna()
].copy()

print("Rows missing image_url:", len(missing_image_rows))

Rows missing image_url: 0


In [17]:
products_df = products_df[
    ~products_df["id"].isin(clean_data.KNOWN_MISSING_IMAGE_IDS)
].reset_index(drop=True)

print("Rows after exclusion:", len(products_df))

Rows after exclusion: 44419


In [18]:
clean_data.run_sanity_checks(products_df)

All sanity checks passed.


## Some final inspections

In [19]:
products_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44419 entries, 0 to 44418
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    44419 non-null  int64  
 1   product_display_name  44419 non-null  object 
 2   brand_name            44419 non-null  object 
 3   gender                44419 non-null  object 
 4   master_category       44419 non-null  object 
 5   sub_category          44419 non-null  object 
 6   article_type          44419 non-null  object 
 7   base_colour           44419 non-null  object 
 8   season                44398 non-null  object 
 9   usage                 44418 non-null  object 
 10  year                  44418 non-null  float64
 11  price                 44419 non-null  float64
 12  discounted_price      44419 non-null  float64
 13  description           44351 non-null  object 
 14  image_url             44419 non-null  object 
 15  pattern            

In [20]:
products_df.isna().sum().sort_values(ascending=False)

length                  40982
neck                    35082
fit                     34338
occasion                33424
sleeve_length           31590
fabric                  28835
pattern                 27926
description                68
season                     21
usage                       1
year                        1
brand_name                  0
product_display_name        0
id                          0
gender                      0
article_type                0
sub_category                0
master_category             0
base_colour                 0
image_url                   0
discounted_price            0
price                       0
search_text                 0
dtype: int64

In [21]:
for i, text in enumerate(
    products_df["search_text"].sample(5, random_state=42),
    start=1
):
    print(f"\n--- Sample {i} ---")
    print(text)


--- Sample 1 ---
Revv Men Steel Pendant Revv Accessories Jewellery Pendant Steel Casual Western revv is an imported stainless steel collection that suits the trendsetting youth of today - cool college goers, young executives and those who like to remain ahead of the pack.

--- Sample 2 ---
Hush Puppies Men Falcon Brown Formal Shoes Hush Puppies Footwear Shoes Formal Shoes Brown Formal If you want the ultimate sensation of comfort for your feet, then these shoes from hush puppies are made for you. These lace-up oxfords with the sporty outsole features herringbone pattern that gives superior grip.Team these with jeans or chinos and tees for a cool look. Upper Leather uppers for durability and comfort Stitch detailing for added style Lace-ups for a snug Fit Lightly padded collar for greater comfort Midsole Cushioned footbed for all day comfort EVA midsole for better shock absorption Outsole Outsole that has elevated herringbone pattern on the forefoot and heel for better grip Slight heel

In [22]:
products_df["year"] = products_df["year"].astype("Int64")
products_df["year"].dtype

Int64Dtype()

In [23]:
clean_data.run_sanity_checks(products_df)

All sanity checks passed.


## Output 

In [24]:
OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "products.parquet"
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

products_df.to_parquet(
    OUTPUT_PATH,
    index=False
)

print(f"Saved to: {OUTPUT_PATH}")

Saved to: /kaggle/working/multi-modal-fashion-ecom/data/processed/products.parquet


In [25]:
loaded_df = pd.read_parquet(OUTPUT_PATH)

print("Shape:", loaded_df.shape)
print("Year dtype:", loaded_df["year"].dtype)

Shape: (44419, 23)
Year dtype: Int64
